# --Silver Layer--


--This notebook reads Employee data from the Bronze layer, performs cleansing, standardization, validation, and writes the curated data to the Silver layer--

In [0]:
import os
import sys

# Current notebook path
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)
print(sys.path[:3])

In [0]:
from src.constants import *
from src.common_functions import *
from src.date_utils import *
from src.validations import *

--Read the bronze employee--

In [0]:
employee_df = spark.table(EMPLOYEE_BRONZE_TABLE)

display(employee_df)

In [0]:
print(employee_df.count())

-Trim and Clean-

In [0]:
employee_df = trim_columns(employee_df)

employee_df = replace_blank_with_null(employee_df)

employee_df = replace_nan_with_null(employee_df)

-Convert Dates-

In [0]:
date_columns = [
    "DOB",
    "Employee_Added",
    "Hire_Date",
    "Rehire_Date",
    "Termination_Date"
]

for column in date_columns:
    employee_df = convert_mixed_date(employee_df, column)

In [0]:
from pyspark.sql.functions import col, to_timestamp

employee_df = (
    employee_df
    .withColumn("Badge_#", col("Badge_#").cast("int"))
    .withColumn("Facility_Code", col("Facility_Code").cast("int"))
    .withColumn("Labor_Position_Code", col("Labor_Position_Code").cast("int"))
    .withColumn("ingestion_timestamp", to_timestamp("ingestion_timestamp"))
)

In [0]:
employee_df.printSchema()

--Remove Duplicates --

In [0]:
employee_df = employee_df.dropDuplicates(["Employee_Code"])

print(f"Rows after removing duplicates: {employee_df.count()}")

--Null Validation--

In [0]:
display(null_summary(employee_df))

In [0]:
print(employee_df.schema)